In [1]:
import torch
import math


class LinearLayer:
    def __init__(self,in_features,out_features,bias=False):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.has_bias= bias
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias = None

    def forward(self,x):
        self.x  = x
        out = self.x @ self.weights.T
        if self.has_bias:
            out = out+self.bias
        
        return out


    def backward(self,grad_out):
        x_shape = self.x.shape
        x_flat = self.x.flatten(0,-2)
        grad_flat = grad_out.flatten(0,-2)

        grad_inputs = grad_flat @ self.weights
        self.weights.grad = grad_flat.T @ x_flat
        if self.has_bias:
            self.bias.grad = grad_flat.sum(dim=0)

        grad_inputs = grad_inputs.reshape(x_shape)
        return grad_inputs


class LayerNorm:

    def __init__(self,num_dims,eps):
        self.eps = eps
        self.gamma = torch.ones(num_dims))
        self.beta  = torch.zeros(num_dims))


    def forward(self,x):
        self.avg = torch.mean(x,dim=-1,keepdim=True)
        self.var = torch.mean( (x - self.avg)**2,dim=-1,keepdim=True )

        self.x_norm = (x - self.avg)/torch.sqrt(self.var+self.eps)
        out = self.x_norm * self.gamma + self.beta
        return out

    
    def backward(self, grad_out):
        self.grad_beta = torch.sum(grad_out, dim=(0, 1))
        self.grad_gamma = torch.sum(grad_out * self.x_norm, dim=(0, 1))
    
        grad_x = grad_out * self.gamma

        return (1.0 / torch.sqrt(self.var + self.eps)) * (
            grad_x
            - torch.mean(grad_x, dim=-1, keepdim=True)
            - self.x_norm * torch.mean(grad_x * self.x_norm, dim=-1, keepdim=True)
        )
        
        

class Relu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0)

    def backward(self,grad_out):
        return grad_out * (self.x > 0)



class Dropout:
    
    def __init__(self,p=0.1,training=True):
        self.p = p
        self.training = training
        self.mask = None
        

    def forward(self,x):
        if self.training:
            self.mask = ((torch.rand_like(x) > self.p).float())/(1.0-self.p)
            return x * self.mask
        else:
            self.mask = None
            return x   

    def backward(self,grad_out):
        if self.training:
            return grad_out * self.mask
        else:
            return grad_out


        
class MyMlp:
    def __init__(self, in_features, hidden_features, out_features):
        self.linear1 = LinearLayer(in_features, hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features, out_features)
        self.dropout = Dropout(p=0.1, training=True)

    def forward(self, x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)
        x = self.dropout.forward(x)
        return x

    def backward(self, grad_out):
        grad_drop = self.dropout.backward(grad_out)
        grad_linear2 = self.linear2.backward(grad_drop)
        grad_relu = self.relu.backward(grad_linear2)
        grad_linear1 = self.linear1.backward(grad_relu)
        return grad_linear1


class softmax:

    def forward(self,scores):
        max_score = torch.max(scores,dim=-1,keepdim=True).values
        scores_exp = torch.exp(scores-max_score)
        self.scores_sum = scores_exp.sum(dim=-1,keepdim=True)
        self.out = scores_exp/self.scores_sum
        return self.out

    
    def backward(self,grad_attn):

        sum_term = (grad_attn *self.out ).sum(dim=-1, keepdim=True)
        grad_scores = self.out * (grad_attn - sum_term)

        return grad_scores
         

class CausalSelfAttention:
    def __init__(self,num_dims,num_heads):
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims,num_dims,bias=False)
        self.w_k = LinearLayer(num_dims,num_dims,bias=False)
        self.w_v = LinearLayer(num_dims,num_dims,bias=False)

        self.proj_out = LinearLayer(num_dims,num_dims,bias=True)
        self.softmax = softmax()

        self.attn_drop = Dropout(p=0.1,training=True)
        self.resid_drop = Dropout(p=0.1,training=True)

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        self.Q = Q
        self.K = K
        self.V = V
        
        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool,device=scores.device),diagonal=1)

        scores = scores.masked_fill(masks,float("-inf"))

        self.attn_scores = self.softmax.forward(scores)
        self.attn_scores = self.attn_drop.forward(self.attn_scores)
        self.out = self.attn_scores @ self.V
        self.out = self.out.transpose(1,2).contiguous().view(B,T,D)
        self.out = self.proj_out(self.out)
        self.out = self.resid_drop.forward(self.out)
        return self.out

    def backward(self,grad_out):
        B,T,D = grad_out.shape
        grad_resid = self.resid_drop.backward(grad_out)
        grad_proj = self.proj_out.backward(grad_resid)
        grad_proj = grad_proj.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        
        grad_attn = grad_proj @ self.V.transpose(-2,-1)
        
        grad_v = self.attn_scores.transpose(-2,-1) @ grad_proj
        
        grad_probs = self.attn_drop.backward(grad_attn)
        grad_scores = self.softmax.backward(grad_probs)   

        scale = 1.0 / math.sqrt(self.head_dims)
        grad_scores = grad_scores * scale
        
        grad_q = grad_scores @ self.K
        grad_k = grad_scores.transpose(-2,-1) @ self.Q
        

        grad_Q = grad_q.transpose(1, 2).contiguous().view(B, T, D)
        grad_K = grad_k.transpose(1, 2).contiguous().view(B, T, D)
        grad_V = grad_v.transpose(1, 2).contiguous().view(B, T, D)

        grad_x_q = self.w_q.backward(grad_Q)
        grad_x_k = self.w_k.backward(grad_K)
        grad_x_v = self.w_v.backward(grad_V)

        grad_x = grad_x_q+grad_x_k+grad_x_v

        return grad_x

    def parameters(self):
        return [self.w_q.weights, self.w_k.weights, self.w_v.weights, self.proj_out.weights, self.proj_out.bias]
        

        
        

# Dropout & Layer Normalization

We use a special method called **Dropout**. We are often encouraged to use Dropout to randomly deactivate a percentage of the neurons, because some neurons can be lazy enough to work and try to copy other neurons' work.

For example:
- $y_1 \to$ feature of neuron 1
- $y_2 \to$ feature of neuron 2
- $y_3 \to y_1 + y_2$

Here $y_3$ isn't actually creating a new feature—it is just using other neurons' work, so we can drop (fuse) this neuron. But in the model, we can't know beforehand which neuron is being redundant, so we drop them randomly to force them to learn features by themselves.



## Where do we actually use Dropout?

### 1. Attention (After Softmax)

Okay wait, why do we need to use it after Softmax? Because there is a clear reason for that:

If you use it before Softmax, those zeroed values become $0$, and $e^0 = 1$ at Softmax—they will revive using reincarnation jutsu 😂️... so we don't want them coming back alive!

After Softmax, we have already done the causal masking for the next tokens so probabilities sum up to a perfect 1. This is the perfect place to use Dropout.

Because we are about to multiply with $V$ (which holds the real content), before the model learns to fetch $V$, we need to make sure the model uses all feature dimensions properly. So we use Dropout at that place.

It prevents the model from obsessing over just one specific token in the context. By randomly zeroing out attention connections, it forces the query token to learn to attend to multiple different context tokens.


### 2. Residual Connection (Before Residual Addition)

And the next position where we actually apply Dropout is just before adding the residual output back to the original tokens.

You might encounter these in the future as we step down a few blocks, so you can skip this part and come back later too. We will soon understand why we actually mix them up (spoilers: because we still need the original $x$ alongside predictions to help the model predict words better).


### 3. MLP Stage

And at the MLP stage, we use Dropout before adding back to $x$.


### 4. Model Entrance (After Embeddings)

At the very entrance of the model, right after combining token embeddings and positional embeddings:

$$x = \text{tok\_emb} + \text{pos\_emb}$$

$$x = \text{dropout}(x)$$



## Layer Normalization (LayerNorm)

Now coming to the real LayerNorm work. The reason for LayerNorm is to ensure that activations don't either blow up or vanish.

Gemini generated this example so well, check this out:


### The Microphone and Amplifier Problem

Imagine a chain of 12 speakers and amplifiers in a room:
- The first amplifier boosts the audio signal by just $1.2\times$.
- The next amplifier boosts it by $1.2\times$.
- By the 12th amplifier, the sound volume is $1.2^{12} \approx 8.9\text{x}$ louder.
- If one amp slips and boosts by $2\times$, $2^{12} = 4096\text{x}$. The sound turns into deafening screeches and distortion.

In a deep neural network, every layer multiplies matrices and adds residual vectors. As activations travel deeper through the network:
- Without control, numbers either blow up toward infinity (**exploding activations**) or shrink toward zero (**vanishing activations**).

When values blow up, mathematical operations like $\text{Softmax}$ choke: the largest number gets a probability of $1.0$, all others become $0.0$, and gradients drop dead to zero.

LayerNorm acts as an automatic sound engineer. Right before the signal enters an Attention or MLP block, LayerNorm grabs that specific token vector, centers its volume to 0, and normalizes its loudness to 1.



## Where is it placed? (Pre-LN vs. Post-LN)

In early Transformers (original "Attention is All You Need"), normalization was applied after the residual add (Post-LN):

$$x = \text{LayerNorm}(x + \text{SubLayer}(x))$$

This caused training instability because backpropagating gradients had to fight through normalization at every single residual step.

Modern architectures like GPT-2, GPT-3, and LLaMA use Pre-LN:

$$x = x + \text{SubLayer}(\text{LayerNorm}(x))$$

In Pre-LN:
- **Clean Highway**: The residual stream $x$ remains an untouched, clean highway where gradients can flow from the end of the model all the way to token embeddings without restriction.
- **Safe Input**: Each sub-layer (Attention and MLP) receives an input that is cleanly scaled and normalized so it never receives exploding numbers.



## LayerNorm Concept & Mathematics

In LayerNorm, single tokens have the entire universe. It's like sending a single token, finding the mean and variance on that token, implementing those calculations, and sending out that single token. So this happens for all the tokens inside.


### Backward Pass for LayerNorm

Okay, I think we are ready for the backward pass, right? Didn't we?

For backward, we actually receive `grad_out`. The `grad_out` comes in, and checking from the back, the output formula was:

$$\text{out} = \text{self.x\_norm} \cdot \text{self.gamma} + \text{self.beta}$$

Now we need to find the derivative with respect to `x_norm`, `gamma`, and `beta`.

I mean, $\gamma$ and $\beta$ are initially 1s and 0s, so the model slowly learns what to change with them, right?

$$\text{grad\_beta} = \text{grad\_out} \cdot 1$$

So we sum it up, just like the bias:

$$\text{grad\_beta} = \text{torch.sum}(\text{grad\_out}, \text{dim}=(0,1))$$

For $\gamma$:

$$\text{grad\_gamma} = \text{grad\_out} \cdot \hat{x}$$

Again, summing over dimensions $0$ and $1$:

$$\text{self.grad\_gamma} = \text{torch.sum}(\text{grad\_out} \cdot \text{self.x\_norm}, \text{dim}=(0, 1))$$


### Derivation of $(u \cdot v)' = u'v + u v'$ for LayerNorm

Let's split the normalization formula into two parts:

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} = u \cdot v$$

where:
- **Numerator $u$**: $u = x - \mu$
- **Scale factor $v$**: $v = (\sigma^2 + \epsilon)^{-1/2}$

Using the Product Rule for differentiation:

$$\frac{\partial \hat{x}}{\partial x} = u'v + u v' = \left(\frac{\partial u}{\partial x}\right) v + u \left(\frac{\partial v}{\partial x}\right)$$


#### Step 1: Derivative of Numerator $u = x - \mu$ ($u'$ term)
Since $\mu = \frac{1}{D}\sum_j x_j$, differentiating $u_i = x_i - \mu$ gives:

$$\frac{\partial u_i}{\partial x_i} = 1 - \frac{1}{D}$$

So the direct numerator gradient contribution is:

$$u'v = \left(1 - \frac{1}{D}\right) \cdot (\sigma^2 + \epsilon)^{-1/2}$$


#### Step 2: Derivative of Scale Factor $v = (\sigma^2 + \epsilon)^{-1/2}$ ($v'$ term)
Using the Chain Rule through variance $\sigma^2$:

$$\frac{\partial v}{\partial x_i} = \frac{\partial v}{\partial \sigma^2} \cdot \frac{\partial \sigma^2}{\partial x_i}$$

Differentiating $v$ with respect to $\sigma^2$:

$$\frac{\partial v}{\partial \sigma^2} = -\frac{1}{2}(\sigma^2 + \epsilon)^{-3/2}$$

And differentiating $\sigma^2 = \frac{1}{D}\sum_j (x_j - \mu)^2$ with respect to $x_i$:

$$\frac{\partial \sigma^2}{\partial x_i} = \frac{2}{D}(x_i - \mu) = \frac{2}{D} u_i$$

Multiplying these together gives $v'$:

$$\frac{\partial v}{\partial x_i} = -\frac{1}{2}(\sigma^2 + \epsilon)^{-3/2} \cdot \frac{2}{D} u_i = -\frac{1}{D} u_i (\sigma^2 + \epsilon)^{-3/2}$$

Now multiplying by $u_i$ to get $uv'$:

$$u v' = u_i \cdot \left(-\frac{1}{D} u_i (\sigma^2 + \epsilon)^{-3/2}\right) = -\frac{1}{D} u_i^2 (\sigma^2 + \epsilon)^{-3/2} = -\frac{1}{D} \hat{x}_i^2 (\sigma^2 + \epsilon)^{-1/2}$$


#### Step 3: Combining $u'v + uv'$ (Multivariable Interaction)

Combining the direct numerator path ($u'v$), the mean contribution, and the variance path ($uv'$):

$$\frac{\partial \mathcal{L}}{\partial x_i} = \frac{1}{\sqrt{\sigma^2 + \epsilon}} \left( d\hat{x}_i - \frac{1}{D}\sum_{j=1}^D d\hat{x}_j - \hat{x}_i \cdot \frac{1}{D}\sum_{j=1}^D (d\hat{x}_j \cdot \hat{x}_j) \right)$$

Factoring out the shared variance scaling factor $\frac{1}{\sqrt{\sigma^2 + \epsilon}}$ yields the clean, vectorized form:

$$\frac{\partial \mathcal{L}}{\partial x} = \frac{1}{\sqrt{\sigma^2 + \epsilon}} \Big( d\hat{x} - \text{mean}(d\hat{x}) - \hat{x} \odot \text{mean}(d\hat{x} \odot \hat{x}) \Big)$$


### PyTorch Implementation

In PyTorch code, this entire multivariable backward pass evaluates cleanly in three lines without loops:

```python
grad_x = grad_out * self.gamma
dx = (1.0 / torch.sqrt(self.var + self.eps)) * (
    grad_x
    - torch.mean(grad_x, dim=-1, keepdim=True)
    - self.x_norm * torch.mean(grad_x * self.x_norm, dim=-1, keepdim=True)
)
```
